# Notebook C — Dataset, Classification et Explainabilité

**Input :** `results/dataset_final.csv` (sortie de `05_build_dataset.py`)  
**Input :** `models/checkpoint_rf.joblib` ou `checkpoint_xgb.joblib`  
**Output :** figures dans `results/figures/` 

Ce notebook produit :
1. Distribution des classes et des features
2. PCA 2D colorée par groupe clinique
3. KNN graph dans l'espace des features (EEG / EMG / all)
4. Matrice de confusion (GroupKFold cross-validation)
5. Feature importance (permutation)

In [ ]:
# ============================================================
# PARAMÈTRES  ←  À MODIFIER
# ============================================================
DATASET_CSV  = "results/dataset_final.csv"
CKPT_PATH    = "models/checkpoint_rf.joblib"   # rf, xgb ou knn
OUTPUT_FIGS  = "results/figures"

LABEL_COL    = "label_id"
LABEL_STR_COL= "label_str"
ID_COL       = "patient_id"

SAVE_FIGS    = True
FIG_FORMAT   = "pdf"
DPI          = 300
N_SPLITS_CV  = 5
KNN_K        = 5

PALETTE = {
    "SYN":   "#44AA99",
    "Narco": "#F15854",
    "TCSPi": "#BEBEBE",
    "EAI":   "#C9D175",
}
CLASS_COLORS = ["#44AA99", "#F15854", "#BEBEBE", "#C9D175"]  # 0=SYN 1=Narco 2=TCSPi 3=EAI
CLASS_NAMES  = ["SYN", "Narco", "TCSPi", "EAI"]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (confusion_matrix, balanced_accuracy_score,
                              f1_score, classification_report)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from matplotlib.colors import ListedColormap, BoundaryNorm

try:
    import joblib
    JOBLIB_OK = True
except ImportError:
    JOBLIB_OK = False
    print("joblib non disponible — section modèle désactivée")

Path(OUTPUT_FIGS).mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
matplotlib.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
print("Librairies chargées.")

In [ ]:
# ============================================================
# CHARGEMENT DU DATASET
# ============================================================
df = pd.read_csv(DATASET_CSV)
df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

num_cols = [c for c in df.columns
            if c not in [ID_COL, LABEL_COL, LABEL_STR_COL]
            and np.issubdtype(df[c].dtype, np.number)]

y      = df[LABEL_COL].values
groups = df[ID_COL].values

print(f"Dataset : {df.shape} | Features numériques : {len(num_cols)}")
print("Distribution :", df[LABEL_COL].value_counts().sort_index().to_dict())

In [ ]:
# ============================================================
# FIGURE 1 — Distribution des classes
# ============================================================
if LABEL_STR_COL in df.columns:
    counts = df.drop_duplicates(ID_COL)[LABEL_STR_COL].value_counts()
    order  = [c for c in CLASS_NAMES if c in counts.index]
    colors = [PALETTE.get(c, "#888") for c in order]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(order, [counts[g] for g in order], color=colors, edgecolor="white", width=0.6)
    for bar, val in zip(bars, [counts[g] for g in order]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, str(val),
                ha="center", va="bottom", fontsize=11)
    ax.set_title("Répartition des patients par groupe clinique", fontsize=12)
    ax.set_ylabel("Nombre de patients")
    ax.set_xlabel("")
    plt.tight_layout()
    if SAVE_FIGS:
        fig.savefig(Path(OUTPUT_FIGS)/f"class_distribution.{FIG_FORMAT}", dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# PRÉPARATION DES GROUPES DE FEATURES
# ============================================================
eeg_spectral = [c for c in num_cols
                if c.startswith("eeg_") and any(b in c for b in ["_bp_","_rel_","_abs_"])]
eeg_temporal = [c for c in num_cols
                if c.startswith("eeg_") and c not in eeg_spectral]
eeg_all      = sorted(set(eeg_spectral + eeg_temporal))
emg_feats    = [c for c in num_cols if c.startswith("emg_")]
all_feats    = num_cols

feature_groups = {
    "All features":  all_feats,
    "EEG (all)":     eeg_all,
    "EEG spectral":  eeg_spectral,
    "EEG temporal":  eeg_temporal,
    "EMG":           emg_feats,
}

print("=== Groupes de features ===")
for k, v in feature_groups.items():
    print(f"  {k:20s} : {len(v)} features")

In [ ]:
# ============================================================
# FIGURE 2 — PCA 2D par groupe de features
# ============================================================
cmap = ListedColormap(CLASS_COLORS)
norm = BoundaryNorm(boundaries=np.arange(-0.5, 4.5, 1), ncolors=4)

fig, axes = plt.subplots(2, 3, figsize=(17, 11))
axes = axes.flatten()

for ax, (title, fcols) in zip(axes, feature_groups.items()):
    if len(fcols) < 3:
        ax.set_title(f"{title}\n(insuffisant)")
        ax.axis("off")
        continue

    X = df[fcols].values.astype(float)
    X = np.where(np.isnan(X), np.nanmedian(X, axis=0), X)
    X = StandardScaler().fit_transform(X)

    pca   = PCA(n_components=2, random_state=42)
    Xpca  = pca.fit_transform(X)
    var   = pca.explained_variance_ratio_ * 100

    sc = ax.scatter(Xpca[:,0], Xpca[:,1], c=y, cmap=cmap, norm=norm,
                    s=55, alpha=0.8, edgecolor="k", linewidth=0.3)
    ax.set_title(f"{title}", fontsize=11, fontweight="bold")
    ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")

# Colorbar commune
cb = fig.colorbar(sc, ax=axes, ticks=[0,1,2,3], shrink=0.6, pad=0.02)
cb.ax.set_yticklabels(CLASS_NAMES)
cb.set_label("Groupe clinique")

fig.suptitle("PCA 2D par sous-espace de features", fontsize=14, fontweight="bold")
plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(Path(OUTPUT_FIGS)/f"pca_feature_groups.{FIG_FORMAT}", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 3 — KNN graph (espace original, projection PCA)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
axes = axes.flatten()

for ax, (title, fcols) in zip(axes, feature_groups.items()):
    if len(fcols) < 3:
        ax.set_title(f"{title}\n(insuffisant)")
        ax.axis("off")
        continue

    X = df[fcols].values.astype(float)
    X = np.where(np.isnan(X), np.nanmedian(X, axis=0), X)
    X = StandardScaler().fit_transform(X)

    Xpca = PCA(n_components=2, random_state=42).fit_transform(X)

    nn = NearestNeighbors(n_neighbors=KNN_K+1)
    nn.fit(X)
    neigh = nn.kneighbors(X, return_distance=False)

    ax.scatter(Xpca[:,0], Xpca[:,1], c=y, cmap=cmap, norm=norm,
               s=55, alpha=0.8, edgecolor="k", linewidth=0.3)

    for i in range(len(Xpca)):
        for j in neigh[i][1:]:
            ax.plot([Xpca[i,0], Xpca[j,0]], [Xpca[i,1], Xpca[j,1]],
                    color="gray", alpha=0.07, linewidth=0.7)

    ax.set_title(f"{title} (k={KNN_K})", fontsize=11, fontweight="bold")
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")

cb = fig.colorbar(sc, ax=axes, ticks=[0,1,2,3], shrink=0.6, pad=0.02)
cb.ax.set_yticklabels(CLASS_NAMES)
cb.set_label("Groupe clinique")

fig.suptitle(f"KNN graph (k={KNN_K}) par sous-espace de features", fontsize=14, fontweight="bold")
plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(Path(OUTPUT_FIGS)/f"knn_feature_groups.{FIG_FORMAT}", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 4 — Matrice de confusion (GroupKFold CV)
# ============================================================
if JOBLIB_OK and Path(CKPT_PATH).exists():
    bundle   = joblib.load(CKPT_PATH)
    model    = bundle["model"]
    fcols    = bundle["features"]
    lbl_map  = bundle.get("label_map", None)

    inv_map   = {int(v): k for k, v in lbl_map.items()} if lbl_map else {i: CLASS_NAMES[i] for i in range(4)}
    cn        = [inv_map.get(c, str(c)) for c in sorted(np.unique(y))]

    X       = df[fcols].values.astype(float)
    imputer = SimpleImputer(strategy="median")
    gkf     = GroupKFold(n_splits=N_SPLITS_CV)

    y_true_all, y_pred_all = [], []
    for tr, va in gkf.split(X, y, groups):
        X_tr = imputer.fit_transform(X[tr])
        X_va = imputer.transform(X[va])
        model.fit(X_tr, y[tr])
        y_true_all.append(y[va])
        y_pred_all.append(model.predict(X_va))

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    ba  = balanced_accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    cm  = confusion_matrix(y_true, y_pred, normalize="true")

    print(classification_report(y_true, y_pred, target_names=cn))

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=cn, yticklabels=cn, ax=ax,
                linewidths=0.4, linecolor="white")
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(f"Matrice de confusion ({N_SPLITS_CV}-fold GroupKFold)\n"
                 f"Balanced Acc={ba:.3f}  |  F1-macro={f1:.3f}", fontsize=11)
    plt.tight_layout()
    if SAVE_FIGS:
        fig.savefig(Path(OUTPUT_FIGS)/f"confusion_matrix.{FIG_FORMAT}", dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print(f"Checkpoint introuvable : {CKPT_PATH}\nSection sautée.")

In [ ]:
# ============================================================
# FIGURE 5 — Feature importance (permutation, top 20)
# ============================================================
if JOBLIB_OK and Path(CKPT_PATH).exists():
    # Entraîner sur tout le dataset pour avoir l'importance globale
    X_full   = imputer.fit_transform(df[fcols].values.astype(float))
    model.fit(X_full, y)

    # Importance native (RF/XGB)
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        imp_df = pd.DataFrame({"feature": fcols, "importance": importances})
        imp_df = imp_df.nlargest(20, "importance").sort_values("importance")

        fig, ax = plt.subplots(figsize=(8, 7))
        colors_bar = ["#44AA99" if c.startswith("eeg_") else "#F15854"
                      for c in imp_df["feature"]]
        ax.barh(imp_df["feature"], imp_df["importance"], color=colors_bar, edgecolor="white")
        ax.set_xlabel("Feature importance")
        ax.set_title("Top 20 features (importance native)\n"
                     "■ EEG  ■ EMG", fontsize=12)

        # Légende manuelle
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(facecolor="#44AA99", label="EEG"),
                            Patch(facecolor="#F15854", label="EMG")], loc="lower right")
        plt.tight_layout()
        if SAVE_FIGS:
            fig.savefig(Path(OUTPUT_FIGS)/f"feature_importance_native.{FIG_FORMAT}",
                        dpi=DPI, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("Checkpoint introuvable — feature importance sautée.")